In [ ]:
!pip install -q kaggle
!pip install -q opencv-python-headless
!pip install -q scikit-learn matplotlib seaborn

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
import warnings
warnings.filterwarnings('ignore')
 
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
)
from tensorflow.keras.optimizers import Adam
 
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelBinarizer
 
from PIL import Image
from glob import glob
 
# Confirm GPU is active
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU:", gpus[0].name if gpus else "⚠ No GPU — go to Runtime > Change runtime type > T4 GPU")

In [ ]:
class Config:
    # Paths
    DATA_DIR       = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
    IMG_DIR_1      = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1"
    IMG_DIR_2      = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2"
    METADATA_PATH  = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv"
    MODEL_SAVE     = "/kaggle/working/best_model.keras"
 
    # Image settings
    IMG_SIZE       = 224          # EfficientNetB0 expects 224x224
    CHANNELS       = 3
    INPUT_SHAPE    = (224, 224, 3)
 
    # Training settings
    BATCH_SIZE     = 32
    EPOCHS         = 30           # Early stopping kicks in before this
    LEARNING_RATE  = 1e-4
    FINE_TUNE_LR   = 3e-5         # Lower LR for fine-tuning phase
    FINE_TUNE_AT   = 50          # Unfreeze EfficientNet from this layer onward
 
    # Data split
    VAL_SIZE       = 0.15
    TEST_SIZE      = 0.15
    RANDOM_SEED    = 42
 
    # Class labels
    CLASSES = {
        'mel':   'Melanoma',
        'nv':    'Melanocytic Nevi',
        'bcc':   'Basal Cell Carcinoma',
        'akiec': 'Actinic Keratosis',
        'bkl':   'Benign Keratosis',
        'df':    'Dermatofibroma',
        'vasc':  'Vascular Lesion'
    }
    NUM_CLASSES = len(CLASSES)
 
cfg = Config()
print("Config loaded. Classes:", list(cfg.CLASSES.values()))

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Load metadata CSV
df = pd.read_csv(cfg.METADATA_PATH)
print(f"Total images in metadata: {len(df)}")
print(df.head())
print("\nColumns:", df.columns.tolist())
print("\nClass distribution:")
print(df['dx'].value_counts())
 
# Build full image paths
# Images are split across two folders — we search both
all_image_paths = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob(os.path.join(cfg.IMG_DIR_1, '*.jpg')) +
              glob(os.path.join(cfg.IMG_DIR_2, '*.jpg'))
}
 
df['path'] = df['image_id'].map(all_image_paths)
missing = df['path'].isnull().sum()
print(f"\nImages found: {len(df) - missing} / {len(df)}")
df = df.dropna(subset=['path'])
 
# Map short labels → readable names
df['label'] = df['dx'].map(cfg.CLASSES)
 
# ── Visualization: Class Distribution ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('HAM10000 Dataset Overview', fontsize=15, fontweight='bold')
 
# Bar chart
counts = df['dx'].value_counts()
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22']
axes[0].bar([cfg.CLASSES[k] for k in counts.index], counts.values, color=colors)
axes[0].set_title('Class Distribution (Imbalanced)', fontweight='bold')
axes[0].set_xticklabels([cfg.CLASSES[k] for k in counts.index], rotation=30, ha='right')
axes[0].set_ylabel('Number of Images')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')
 
# Pie chart
axes[1].pie(counts.values, labels=[cfg.CLASSES[k] for k in counts.index],
            colors=colors, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class Proportions', fontweight='bold')
 
plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("⚠ Notice: Melanocytic Nevi (nv) dominates at ~67% — class imbalance must be handled")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Sample Images Per Class', fontsize=14, fontweight='bold')
 
for idx, (dx, label) in enumerate(cfg.CLASSES.items()):
    row, col = divmod(idx, 4)
    sample = df[df['dx'] == dx].sample(1).iloc[0]
    img = cv2.imread(sample['path'])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (cfg.IMG_SIZE, cfg.IMG_SIZE))
    axes[row][col].imshow(img_resized)
    axes[row][col].set_title(f'{label}\n({dx})', fontweight='bold', fontsize=9)
    axes[row][col].axis('off')
 
# Hide the unused 8th subplot
axes[1][3].axis('off')
plt.tight_layout()
plt.savefig('/content/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Encode labels as integers
label_to_idx = {dx: i for i, dx in enumerate(cfg.CLASSES.keys())}
idx_to_label = {i: dx for dx, i in label_to_idx.items()}
idx_to_name  = {i: cfg.CLASSES[dx] for dx, i in label_to_idx.items()}

df['label_idx'] = df['dx'].map(label_to_idx)

# Group-aware, stratified split on lesion_id: a lesion can have multiple
# images (different angles/zoom), so a plain row-wise split can leak the
# same lesion into both train and test. StratifiedGroupKFold keeps every
# image of a given lesion_id in a single fold while still trying to
# preserve each class's proportions within every fold.
#
# There's no direct test_size knob, so we cut the data into 20 folds
# (~5% each) and combine folds to land on ~70/15/15: 14 folds → train,
# 3 folds → val, 3 folds → test. Every sample appears in exactly one
# fold's held-out set, so the three groups are disjoint by construction.
N_FOLDS = 20
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=cfg.RANDOM_SEED)
fold_indices = [held_out_idx for _, held_out_idx in
                sgkf.split(df, df['label_idx'], groups=df['lesion_id'])]

train_idx = np.concatenate(fold_indices[0:14])   # 14/20 = 70%
val_idx   = np.concatenate(fold_indices[14:17])  # 3/20  = 15%
test_idx  = np.concatenate(fold_indices[17:20])  # 3/20  = 15%

train_df = df.iloc[train_idx]
val_df   = df.iloc[val_idx]
test_df  = df.iloc[test_idx]

print(f"Train:      {len(train_df)} images")
print(f"Validation: {len(val_df)} images")
print(f"Test:       {len(test_df)} images")
print(f"Total:      {len(train_df) + len(val_df) + len(test_df)} images")

# Verify no lesion_id appears in more than one split
train_lesions = set(train_df['lesion_id'])
val_lesions   = set(val_df['lesion_id'])
test_lesions  = set(test_df['lesion_id'])

assert len(train_lesions & val_lesions) == 0, "Lesion overlap between train and val!"
assert len(train_lesions & test_lesions) == 0, "Lesion overlap between train and test!"
assert len(val_lesions & test_lesions) == 0, "Lesion overlap between val and test!"
print(f"\nLesion overlap check passed — unique lesions: "
      f"{len(train_lesions)} train / {len(val_lesions)} val / {len(test_lesions)} test")

# Per-class counts in each split, to confirm stratification held up
print("\nPer-class counts by split:")
for split_name, split_df in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    counts = split_df['label_idx'].value_counts().reindex(range(cfg.NUM_CLASSES), fill_value=0)
    print(f"  {split_name}:")
    for idx, cnt in counts.items():
        print(f"    {idx_to_name[idx]:25s}: {cnt}")

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['label_idx']),
    y=train_df['label_idx']
)
class_weights = dict(enumerate(class_weights_array))
print("\nClass weights (higher = rarer class):")
for idx, weight in class_weights.items():
    print(f"  {idx_to_name[idx]:25s}: {weight:.3f}")

In [ ]:
# What is augmentation?
# We artificially create variations of training images so the model
# doesn't just memorize — it learns to generalize.
# Val/Test data gets NO augmentation (we want real-world accuracy).

def preprocess_image(path, target_size=(cfg.IMG_SIZE, cfg.IMG_SIZE)):
    """Load an image as float32 in [0, 255]. EfficientNetB0 rescales internally."""
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, target_size)
    img = img.astype(np.float32)
    return img


def load_dataset(dataframe, augment=False, batch_size=cfg.BATCH_SIZE, shuffle=True):
    """
    Build a tf.data pipeline from a dataframe.
    Much faster than loading images in a loop.
    """
    paths  = dataframe['path'].values
    labels = dataframe['label_idx'].values

    def load_and_preprocess(path, label):
        # Read image bytes → decode → resize → scale to [0,1]. The
        # augmentation ops below (brightness/contrast/hue/saturation)
        # are tuned for [0,1] inputs, so we work in that range first.
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [cfg.IMG_SIZE, cfg.IMG_SIZE])
        img = tf.cast(img, tf.float32) / 255.0

        if augment:
            # Randomly flip left-right (50% chance)
            img = tf.image.random_flip_left_right(img)
            # Randomly flip up-down (50% chance)
            img = tf.image.random_flip_up_down(img)
            # Slight brightness shift
            img = tf.image.random_brightness(img, max_delta=0.15)
            # Slight contrast shift
            img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
            # Slight hue shift (important for skin tone variation)
            img = tf.image.random_hue(img, max_delta=0.05)
            # Slight saturation shift
            img = tf.image.random_saturation(img, lower=0.85, upper=1.15)
            # Clip values back to [0,1] after adjustments
            img = tf.clip_by_value(img, 0.0, 1.0)

        # Scale back up to [0, 255] as the final step — applies to the
        # augmented train path and the untouched val/test path alike.
        # EfficientNetB0 rescales internally and expects raw-range input.
        img = img * 255.0

        label = tf.one_hot(label, depth=cfg.NUM_CLASSES)
        return img, label

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(paths), seed=cfg.RANDOM_SEED)
    dataset = dataset.map(load_and_preprocess,
                          num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)  # Load next batch while GPU trains
    return dataset


train_ds = load_dataset(train_df, augment=True,  shuffle=True)
val_ds   = load_dataset(val_df,   augment=False, shuffle=False)
test_ds  = load_dataset(test_df,  augment=False, shuffle=False)

# Preview one augmented batch
for images, labels in train_ds.take(1):
    print(f"Batch shape: {images.shape}")  # (32, 224, 224, 3)
    print(f"Label shape: {labels.shape}")  # (32, 7)
    print(f"Pixel range: [{images.numpy().min():.2f}, {images.numpy().max():.2f}]")

In [ ]:
# See what augmented images look like
sample_img_path = train_df.sample(1).iloc[0]['path']
original = preprocess_image(sample_img_path)
 
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Original vs Augmented Versions of Same Image', fontsize=13, fontweight='bold')
axes[0][0].imshow(original/ 255.0)
axes[0][0].set_title('Original', fontweight='bold')
axes[0][0].axis('off')
 
# Generate 9 augmented versions
aug_fn = load_dataset(
    pd.concat([train_df[train_df['path'] == sample_img_path]] * 9),
    augment=True, batch_size=9, shuffle=False
)
for batch_imgs, _ in aug_fn.take(1):
    for i, img in enumerate(batch_imgs):
        row, col = divmod(i + 1, 5)
        axes[row][col].imshow(img.numpy() / 255.0)
        axes[row][col].set_title(f'Augmented {i+1}', fontsize=9)
        axes[row][col].axis('off')
 
plt.tight_layout()
plt.savefig('/content/augmentation_preview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# WHAT IS TRANSFER LEARNING?
# EfficientNetB0 was trained on 1.2 million general images (ImageNet).
# It already knows edges, textures, shapes, patterns.
# We FREEZE those weights and add our own layers on top,
# training only our custom head first (faster, less data needed).
# Then we UNFREEZE the top layers and fine-tune with a tiny LR.
#

def build_model(num_classes=cfg.NUM_CLASSES, trainable_base=False):
    """
    EfficientNetB0 base + custom classification head, built as a single
    flat graph (base layers attached directly to `inputs` via
    `input_tensor`, not wrapped as a nested sub-model). This keeps every
    EfficientNet layer — including `top_conv`, the last conv layer —
    directly reachable via model.get_layer(...), which a nested
    sub-model would hide and which Grad-CAM needs.

    Architecture:
      Input (224x224x3)
        └── EfficientNetB0 layers (pretrained, 5.3M params)
              └── GlobalAveragePooling2D
                    └── BatchNormalization
                          └── Dropout(0.4)
                                └── Dense(256, swish)
                                      └── BatchNormalization
                                            └── Dropout(0.3)
                                                  └── Dense(7, softmax)
    """
    inputs = keras.Input(shape=cfg.INPUT_SHAPE)

    # ── Base model, built directly on `inputs` (flat, not nested) ──
    base = EfficientNetB0(
        include_top=False,          # Remove ImageNet classification head
        weights='imagenet',         # Use pretrained weights
        input_tensor=inputs
    )
    base.trainable = trainable_base  # Freeze during Phase 1

    # ── Custom classification head ──
    x = base.output

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(256, activation='swish',
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    return model, base


# Build Phase 1 model (frozen base)
model, base_model = build_model(trainable_base=False)

model.compile(
    optimizer=Adam(learning_rate=cfg.LEARNING_RATE),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.0, alpha=0.25),
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc', multi_label=False),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)

model.summary()
total_params     = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {total_params - trainable_params:,}")

In [ ]:
callbacks = [
    # Save the best model (by val_auc, not just accuracy)
    ModelCheckpoint(
        filepath=cfg.MODEL_SAVE,
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    # Stop training if val_loss doesn't improve for 8 epochs
    EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    # Halve learning rate if val_loss plateaus for 4 epochs
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    )
]
 
print("Callbacks ready.")

In [ ]:
# Train only the custom head.
# Fast because we're updating ~330K params, not 5.3M.
 
print("=" * 60)
print("PHASE 1: Training custom head (base frozen)")
print("=" * 60)
 
history_phase1 = model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)
 
print("\nPhase 1 complete.")
print(f"Best val_auc: {max(history_phase1.history['val_auc']):.4f}")

In [ ]:
# Now unfreeze top ~40 layers of EfficientNetB0 and train
# with a much smaller learning rate (10x smaller).
# This lets the model adapt its low-level features to skin images.
 
print("=" * 60)
print("PHASE 2: Fine-tuning (unfreezing top EfficientNet layers)")
print("=" * 60)
 
# Unfreeze top layers. `base_model`'s layers are the same layer objects
# used directly inside the flat `model` graph (no nested sub-model call
# with a hardcoded training=False), so this freeze/unfreeze actually
# takes effect on BatchNorm behavior during fine-tuning.
base_model.trainable = True
# Keep layers below fine_tune_at frozen
for layer in base_model.layers[:cfg.FINE_TUNE_AT]:
    layer.trainable = False
# Even among the unfrozen top layers, keep BatchNorm frozen — fine-tuning
# BN running statistics at this batch size destabilizes training.
for layer in base_model.layers[cfg.FINE_TUNE_AT:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
 
fine_tune_trainable = sum(
    [tf.size(w).numpy() for w in model.trainable_weights]
)
print(f"Trainable parameters after unfreeze: {fine_tune_trainable:,}")
 
# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=cfg.FINE_TUNE_LR),
    loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.0, alpha=0.25),
    metrics=[
        'accuracy',
        keras.metrics.AUC(name='auc', multi_label=False),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall')
    ]
)
 
# Refresh callbacks with same checkpoint path
callbacks_phase2 = [
    ModelCheckpoint(
        filepath=cfg.MODEL_SAVE,
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-8,
        verbose=1
    )
]
 
history_phase2 = model.fit(
    train_ds,
    epochs=cfg.EPOCHS,
    validation_data=val_ds,
    class_weight=class_weights,
    callbacks=callbacks_phase2,
    verbose=1
)
 
print("\nPhase 2 complete.")
print(f"Best val_auc: {max(history_phase2.history['val_auc']):.4f}")

In [ ]:
def plot_training_history(h1, h2):
    """Merge Phase 1 + Phase 2 history and plot."""
    # Merge histories
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
 
    epochs = range(1, len(merged['accuracy']) + 1)
    phase1_end = len(h1.history['accuracy'])
 
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    fig.suptitle('Training History', fontsize=15, fontweight='bold')
 
    metrics = [
        ('accuracy',  'val_accuracy',  'Accuracy',  '#3498db'),
        ('auc',       'val_auc',       'AUC',       '#e74c3c'),
        ('loss',      'val_loss',      'Loss',      '#2ecc71'),
        ('precision', 'val_precision', 'Precision', '#f39c12'),
    ]
 
    for ax, (train_m, val_m, title, color) in zip(axes.flat, metrics):
        ax.plot(epochs, merged[train_m], color=color, linewidth=2, label='Train')
        ax.plot(epochs, merged[val_m], color=color, linewidth=2,
                linestyle='--', alpha=0.7, label='Validation')
        ax.axvline(phase1_end, color='gray', linestyle=':', linewidth=1.5,
                   label='Fine-tune starts')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)
 
    plt.tight_layout()
    plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
 
plot_training_history(history_phase1, history_phase2)

In [ ]:
# Load best saved model
best_model = keras.models.load_model(cfg.MODEL_SAVE)
 
# Get predictions on test set
print("Running inference on test set...")
y_pred_probs = best_model.predict(test_ds, verbose=1)  # Shape: (N, 7)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = test_df['label_idx'].values
 
# ── Overall Metrics ──
from sklearn.metrics import accuracy_score, f1_score
acc     = accuracy_score(y_true, y_pred)
f1_mac  = f1_score(y_true, y_pred, average='macro')
f1_wt   = f1_score(y_true, y_pred, average='weighted')
 
print("\n" + "=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print(f"Overall Accuracy:       {acc:.4f}  ({acc*100:.2f}%)")
print(f"Macro F1 Score:         {f1_mac:.4f}")
print(f"Weighted F1 Score:      {f1_wt:.4f}")
 
# Per-class report
class_names = [cfg.CLASSES[idx_to_label[i]] for i in range(cfg.NUM_CLASSES)]
print("\nPer-Class Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100  # Row-normalized %
 
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('Confusion Matrix — Test Set', fontsize=14, fontweight='bold')
 
short_names = [cfg.CLASSES[idx_to_label[i]][:12] for i in range(cfg.NUM_CLASSES)]
 
# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=short_names, yticklabels=short_names,
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Raw Counts', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].tick_params(axis='x', rotation=30)
 
# Percentage
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Oranges',
            xticklabels=short_names, yticklabels=short_names,
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Row-Normalized (%)', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].tick_params(axis='x', rotation=30)
 
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
lb = LabelBinarizer()
y_true_bin = lb.fit_transform(y_true)  # Shape: (N, 7) one-hot
 
fig, ax = plt.subplots(figsize=(11, 8))
ax.set_title('ROC Curves — One-vs-Rest Per Class', fontsize=13, fontweight='bold')
 
colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c','#e67e22']
 
for i, (name, color) in enumerate(zip(class_names, colors)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{name[:20]} (AUC = {roc_auc:.3f})')
 
# Macro-average AUC
macro_auc = roc_auc_score(y_true_bin, y_pred_probs,
                          multi_class='ovr', average='macro')
 
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.text(0.05, 0.95, f'Macro AUC: {macro_auc:.4f}',
        transform=ax.transAxes, fontsize=12, fontweight='bold',
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
 
plt.tight_layout()
plt.savefig('/content/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Macro AUC: {macro_auc:.4f}")

In [ ]:
# ================================================================
# CELL 18 — Grad-CAM on the last conv layer (top_conv)
# ================================================================
# Works now because `best_model` is a flat graph (see build_model):
# `top_conv` is directly reachable via model.get_layer(), with no
# nested sub-model in the way.

def make_gradcam_heatmap(img_batch, model, last_conv_layer_name='top_conv', pred_index=None):
    """Compute a Grad-CAM heatmap for a single preprocessed image batch."""
    grad_model = Model(
        model.inputs,
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_batch)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)


def overlay_gradcam(img_uint8, heatmap, alpha=0.4):
    """Overlay a Grad-CAM heatmap (0-1 float array) on an RGB uint8 image."""
    heatmap_resized = cv2.resize(heatmap, (img_uint8.shape[1], img_uint8.shape[0]))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlay = img_uint8.astype(np.float32) * (1 - alpha) + heatmap_color.astype(np.float32) * alpha
    return np.uint8(np.clip(overlay, 0, 255))


# Pick six test images spanning six different classes
sample_rows = (test_df.groupby('label_idx', group_keys=False)
                       .apply(lambda g: g.sample(1, random_state=cfg.RANDOM_SEED))
                       .head(6))

fig, axes = plt.subplots(2, 6, figsize=(22, 8))
fig.suptitle('Grad-CAM — Last Conv Layer (top_conv)', fontsize=14, fontweight='bold')

for col, (_, row) in enumerate(sample_rows.iterrows()):
    img = cv2.imread(row['path'])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (cfg.IMG_SIZE, cfg.IMG_SIZE)).astype(np.float32)

    img_batch = np.expand_dims(img, axis=0)  # float32, [0, 255], matches training preprocessing
    heatmap, pred_idx = make_gradcam_heatmap(img_batch, best_model, last_conv_layer_name='top_conv')
    overlay = overlay_gradcam(img.astype(np.uint8), heatmap)

    true_name = cfg.CLASSES[idx_to_label[row['label_idx']]]
    pred_name = cfg.CLASSES[idx_to_label[pred_idx]]

    axes[0][col].imshow(img.astype(np.uint8))
    axes[0][col].set_title(f"{true_name}", fontsize=9, fontweight='bold')
    axes[0][col].axis('off')

    axes[1][col].imshow(overlay)
    axes[1][col].set_title(f"Grad-CAM\n(pred: {pred_name})", fontsize=8)
    axes[1][col].axis('off')

plt.tight_layout()
plt.savefig('/content/gradcam_examples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 19 - Skipping skin tone analysis for now
print("Skipping bias analysis. Proceeding to save model...")

In [ ]:
# Cell 20 - Load and save model
import json

# Reload the best model from where it was saved during training
best_model = tf.keras.models.load_model('/kaggle/working/best_model.keras')
print("Model loaded successfully.")

# Save TFLite version for the app
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('/kaggle/working/skin_cancer_model.tflite', 'wb') as f:
    f.write(tflite_model)

# Save class labels
labels_dict = {str(i): idx_to_label[i] for i in range(cfg.NUM_CLASSES)}
with open('/kaggle/working/class_labels.json', 'w') as f:
    json.dump(labels_dict, f, indent=2)

print("Files saved:")
print("  /kaggle/working/skin_cancer_model.tflite")
print("  /kaggle/working/class_labels.json")
print(f"  TFLite size: {os.path.getsize('/kaggle/working/skin_cancer_model.tflite')/1e6:.1f} MB")